# Crystal Visualization Workflows

This notebook demonstrates the current VESTA-like 3D crystal-visualization surface
for publication-quality structure views with atoms, bonds, cell overlays, bounded
planes, and crystallographic directions.


In [ ]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_orientations,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_context():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
    lattice = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0, crystal_frame=crystal)
    unit_cell = UnitCell(
        lattice=lattice,
        sites=(
            AtomicSite("A1", "Cu", np.array([0.0, 0.0, 0.0])),
            AtomicSite("A2", "Cu", np.array([0.0, 0.5, 0.5])),
            AtomicSite("A3", "Cu", np.array([0.5, 0.0, 0.5])),
            AtomicSite("A4", "Cu", np.array([0.5, 0.5, 0.0])),
        ),
    )
    phase = Phase(
        "fcc_demo",
        lattice=lattice,
        symmetry=symmetry,
        crystal_frame=crystal,
        unit_cell=unit_cell,
    )
    return crystal, specimen, map_frame, detector, lab, phase


def make_nacl_phase():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    lattice = Lattice(5.6402, 5.6402, 5.6402, 90.0, 90.0, 90.0, crystal_frame=crystal)
    symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
    unit_cell = UnitCell(
        lattice=lattice,
        sites=(
            AtomicSite("Cl1", "Cl", np.array([0.0, 0.0, 0.0])),
            AtomicSite("Cl2", "Cl", np.array([0.0, 0.5, 0.5])),
            AtomicSite("Cl3", "Cl", np.array([0.5, 0.0, 0.5])),
            AtomicSite("Cl4", "Cl", np.array([0.5, 0.5, 0.0])),
            AtomicSite("Na1", "Na", np.array([0.5, 0.0, 0.0])),
            AtomicSite("Na2", "Na", np.array([0.0, 0.5, 0.0])),
            AtomicSite("Na3", "Na", np.array([0.0, 0.0, 0.5])),
            AtomicSite("Na4", "Na", np.array([0.5, 0.5, 0.5])),
        ),
    )
    return Phase(
        "NaCl",
        lattice=lattice,
        symmetry=symmetry,
        crystal_frame=crystal,
        unit_cell=unit_cell,
    )


def load_zr_hcp_phase():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    return Phase.from_cif(Path("fixtures/phases/zr_hcp/phase.cif"), crystal_frame=crystal)


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


## Publication-Style NaCl Example

This first scene shows how one viewer can carry atoms, bonds, repeated unit cells,
bounded Miller planes, and labeled directions at the same time.


In [ ]:
phase = make_nacl_phase()

scene = build_crystal_scene(
    phase,
    repeats=(2, 2, 2),
    show_unit_cells=True,
    cell_overlays=(
        CrystalCellOverlay(
            kind="parallelepiped",
            anchor_fractional=np.array([0.0, 0.0, 0.0]),
            color="#0f172a",
            alpha=0.95,
            linewidth=1.1,
        ),
    ),
    plane_overlays=(
        CrystalPlaneOverlay(
            plane=CrystalPlane(MillerIndex([1, 1, 1], phase=phase), phase=phase),
            label_indices=(1, 1, 1),
            color="#f97316",
            alpha=0.18,
            annotation_style=PlaneAnnotationStyle(fontsize=10.5),
        ),
        CrystalPlaneOverlay(
            plane=CrystalPlane(MillerIndex([1, -2, 0], phase=phase), phase=phase),
            label_indices=(1, -2, 0),
            color="#14b8a6",
            alpha=0.22,
            annotation_style=PlaneAnnotationStyle(fontsize=10.5),
        ),
    ),
    direction_overlays=(
        CrystalDirectionOverlay(
            direction=CrystalDirection(np.array([1.0, 1.0, 1.0]), phase=phase),
            anchor_fractional=np.array([0.0, 0.0, 0.0]),
            label_indices=(1, 1, 1),
            color="#2563eb",
            annotation_style=DirectionAnnotationStyle(fontsize=10.5),
        ),
        CrystalDirectionOverlay(
            direction=CrystalDirection(np.array([1.0, -2.0, 0.0]), phase=phase),
            anchor_fractional=np.array([1.0, 1.0, 1.0]),
            label_indices=(1, -2, 0),
            color="#7c3aed",
            annotation_style=DirectionAnnotationStyle(fontsize=10.5),
        ),
    ),
    theme="presentation",
    style_overrides=publication_crystal_style(),
)

print("atoms:", len(scene.atoms))
print("bonds:", len(scene.bonds))
print("cells:", len(scene.cells))
print("planes:", len(scene.planes))
print("directions:", len(scene.directions))


In [ ]:
figure = plot_crystal_structure_3d(
    scene,
    projection="persp",
    style_overrides=publication_crystal_style(),
)
figure


The viewer renders geometry already defined in the canonical structure model. It
does not redefine the lattice, the unit cell, or the plane conventions.


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    destination = Path(tmpdir) / "nacl_publication_view.png"
    figure.savefig(destination, dpi=300, bbox_inches="tight")
    print(destination)


## Hexagonal-Axis Zr Example

The next scene adds the pedagogical hexagonal prism so the basal symmetry is easier
to teach while keeping the canonical lattice semantics unchanged.


In [ ]:
zr_hcp = load_zr_hcp_phase()

zr_scene = build_crystal_scene(
    zr_hcp,
    repeats=(2, 2, 2),
    show_unit_cells=True,
    cell_overlays=(
        CrystalCellOverlay(
            kind="hexagonal_prism",
            anchor_fractional=np.array([1.0, 1.0, 0.0]),
            color="#0f766e",
            alpha=0.95,
            linewidth=1.4,
            show_faces=True,
            face_alpha=0.08,
        ),
    ),
    plane_overlays=(
        CrystalPlaneOverlay(
            plane=CrystalPlane.from_miller_bravais((1, 1, -2, 1), phase=zr_hcp),
            label_indices=(1, 1, -2, 1),
            color="#ef4444",
            alpha=0.20,
        ),
        CrystalPlaneOverlay(
            plane=CrystalPlane.from_miller_bravais((0, 0, 0, 1), phase=zr_hcp),
            label_indices=(0, 0, 0, 1),
            color="#0ea5e9",
            alpha=0.16,
        ),
    ),
    direction_overlays=(
        CrystalDirectionOverlay(
            direction=CrystalDirection.from_miller_bravais((2, -1, -1, 0), phase=zr_hcp),
            anchor_fractional=np.array([0.0, 0.0, 0.0]),
            label_indices=(2, -1, -1, 0),
            color="#f59e0b",
        ),
        CrystalDirectionOverlay(
            direction=CrystalDirection.from_miller_bravais((0, 0, 0, 1), phase=zr_hcp),
            anchor_fractional=np.array([0.2, 0.2, 0.0]),
            label_indices=(0, 0, 0, 1),
            color="#2563eb",
        ),
    ),
    theme="journal",
    style_overrides=publication_crystal_style(),
)

print("hex cells:", len(zr_scene.cells))
print("planes:", len(zr_scene.planes))
print("directions:", len(zr_scene.directions))


In [ ]:
zr_figure = plot_crystal_structure_3d(
    zr_scene,
    projection="persp",
    style_overrides=publication_crystal_style(),
)
zr_figure
